# Mapping the Potential Destructive Power of Wildfires Using Machine Learning
---
## Module 6: *Class Balancing*
##### Version Number: 2.0
---
### Contents  
> 1. Data Preparation
> 2. Automatic Class Balancing 
> 3. Export File
---
### Notes

**WARNING** this module is computation heavy
- Start with a **baseline model** for comparison.
- Test with multi-classification **tree-based models** (Random Forest, XGBoost) and KNN.
- Test and evaluate multiple class balancing strategies (No sampling, Oversampling, Undersampling
- Compare strategies analyzing average F1 score among all classes

---
### Inputs
- `X`
- `y`

---
### Outputs  

`best_strategy` - dataframe recording best sampling strategy for each method

---
### User Created Dependencies

In [1]:
# Add the parent directory to the system path so "src" can be found
import sys
import os

sys.path.append(os.path.abspath(os.path.join('..')))

# user built utilities
from src.model_utils import gen_report
from src.model_utils import kfold

---
### Third Party Dependencies

In [2]:
# Core Python libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing and modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# Resampling tools
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

# Style
sns.set(style='whitegrid')
plt.rcParams["figure.dpi"] = 100

---

## 1. Prepare Data for Modeling - Scaling, Splitting, and Resampling

In [3]:
# Load processed feature and label data
X = pd.read_csv('../data/processed/X_scaled.csv')
y = pd.read_csv('../data/processed/y_reduced.csv').squeeze()

### 1.2 Split Data

In [4]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

---

## 2. Automatic Class Balancing

To address the extreme inbalance in the dataset, multiple strategies are explored.
- **In Method Balancing** is used when applicable 
- **RandomUnderSampler** will remove random members of the majority class (Low severity) until they are balanced. This creates a much smaller dataset to model.
- **SMOTE (Synthetic Minority Over-sampling Technique)** will generate synthetic members of the minority classes. Introduces potential noise by synthetic sampling

### 2.1 Utility functions

A function to perform both in and out of sample testing with kfold crossvalidation and generate a report of metrics.

In [5]:
def class_balancing(X_train, y_train, X_test, y_test, model, sampling_strategy='No_balance'):
    
    if sampling_strategy == 'Undersampling':
        rus = RandomUnderSampler(sampling_strategy='auto', random_state=14)
        X_train, y_train = rus.fit_resample(X_train, y_train)
   
    if sampling_strategy == 'Oversampling':
        smote = SMOTE()
        X_train, y_train = smote.fit_resample(X_train, y_train)
          
    reports = kfold(X_train, y_train, model)    
    Train_metrics = gen_report(reports)

    # Retrain on full training set
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    reports = [classification_report(y_test, y_pred, output_dict=True)]
    Test_metrics = gen_report(reports)

    # Add context columns
    for df in [Train_metrics, Test_metrics]:
        df['Phase'] = 'Train' if df is Train_metrics else 'Test'
        df['Model'] = model.__class__.__name__
        df['Balancing'] = sampling_strategy

    # Combine and reorder
    combined_metrics = pd.concat([Train_metrics, Test_metrics], axis=0)
    combined_metrics = combined_metrics.reset_index().rename(columns={'index': 'Class'})

    return combined_metrics

### 2.2 Define default machine learning models

In [6]:
default_rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
)

In [7]:
default_nn_model = MLPClassifier(
    hidden_layer_sizes= (64, 32),
    activation='relu',
    solver='adam',
    alpha=0.0001,                 
    batch_size=32,
    learning_rate='constant',
    max_iter=500, 
    early_stopping=True,
    random_state=14,
    verbose=False 
)

In [8]:
default_xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
) 

 ###  2.3 Compare Sampling Methods

In [9]:
models = {
    'RF': default_rf_model,
    'NN': default_nn_model,
    'XGB': default_xgb_model
}

sampling_strategies = ['Undersampling','No_balance','Oversampling']

all_results = []
counter = 1

for strategy in sampling_strategies:
    for name, model in models.items():
        print("running", strategy, counter, "of 9...")
        counter = counter + 1
        result = class_balancing(
            X_train, y_train, X_test, y_test,
            model,
            sampling_strategy=strategy
        )
        result['Model_Label'] = name
        all_results.append(result)

# Combine into one DataFrame
all_results_df = pd.concat(all_results, axis=0).reset_index(drop=True)

running Undersampling 1 of 9...
running Undersampling 2 of 9...
running Undersampling 3 of 9...
running No_balance 4 of 9...
running No_balance 5 of 9...


C:\Users\dusti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\dusti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\dusti\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\dusti\anaconda3\Lib\site-packag

running No_balance 6 of 9...
running Oversampling 7 of 9...
running Oversampling 8 of 9...
running Oversampling 9 of 9...


### 2.4 Format and display results

In [10]:
strategy_summary = (
    all_results_df[all_results_df['Phase'] == 'Test']
    .groupby(['Balancing','Model_Label'])['F1-Score']
    .mean()
    .reset_index()
    .sort_values(by='F1-Score', ascending=False)
)

strategy_summary_pivot = (
    strategy_summary
    .pivot(index='Model_Label', columns='Balancing', values='F1-Score')
    .sort_values(by='No_balance', ascending=False)
)

strategy_summary_pivot.style.highlight_max(axis=1, color='lightgreen')

Balancing,No_balance,Oversampling,Undersampling
Model_Label,,,
XGB,0.728127,0.721046,0.710293
RF,0.708325,0.704310,0.699614
NN,0.445464,0.386561,0.501319


#### Best Strategies

In [11]:
# Find best balancing strategy per model
best_strategy = strategy_summary_pivot.idxmax(axis=1)

# Combine with model names into a new DataFrame
best_strategy_df = pd.DataFrame({
    'Model_Label': best_strategy.index,
    'Best_Strategy': best_strategy.values
})

In [12]:
best_strategy_df

,Model_Label,Best_Strategy
0,XGB,No_balance
1,RF,No_balance
2,NN,Undersampling


### Detailed results

In [13]:
all_results_df

,Class,Category,Precision,Recall,F1-Score,Phase,Model,Balancing,Model_Label
0,0,0,0.761374,0.734391,0.747605,Train,RandomForestClassifier,Undersampling,RF
1,1,1,0.767315,0.654794,0.706545,Train,RandomForestClassifier,Undersampling,RF
2,2,2,0.743596,0.879234,0.805677,Train,RandomForestClassifier,Undersampling,RF
3,0,0,0.824144,0.733797,0.776351,Test,RandomForestClassifier,Undersampling,RF
4,1,1,0.787945,0.661591,0.719261,Test,RandomForestClassifier,Undersampling,RF
5,2,2,0.459321,0.878461,0.603231,Test,RandomForestClassifier,Undersampling,RF
6,0,0,0.557294,0.556716,0.510413,Train,MLPClassifier,Undersampling,NN
7,1,1,0.655870,0.304713,0.395046,Train,MLPClassifier,Undersampling,NN
8,2,2,0.467681,0.640077,0.528440,Train,MLPClassifier,Undersampling,NN
9,0,0,0.668237,0.635743,0.651585,Test,MLPClassifier,Undersampling,NN


---

## 3. Export File

In [14]:
best_strategy_df.to_csv('../data/processed/best_strategy.csv',index=False)
print("All datasets saved successfully to ../data/processed/")

All datasets saved successfully to ../data/processed/
